# Specific Test VII — Physics-Guided ML: PINN Classifier

**Project**: DeepLense — GSoC 2026
**Task**: Build a Physics-Informed Neural Network (PINN) that uses the gravitational lensing equation in its architecture to improve multi-class classification of lensing images.
**Dataset split**: 90 % train / 10 % val (stratified, seed 42) — 33,750 / 3,750 images.

---

## 1. Physics Background

The **thin-lens equation** (Singular Isothermal Sphere, SIS model):

$$\boldsymbol{\beta} = \boldsymbol{\theta} - \theta_E \, \frac{\boldsymbol{\theta}}{|\boldsymbol{\theta}|}$$

where:
- $\boldsymbol{\beta}$ = source position
- $\boldsymbol{\theta}$ = image position
- $\theta_E$ = Einstein radius (the scale of the Einstein ring)

For an on-axis source ($\boldsymbol{\beta} = 0$) a perfect ring forms at $|\boldsymbol{\theta}| = \theta_E$.

Real images deviate from the SIS prediction according to substructure type:
- **No substructure** → smooth ring; residual $\approx 0$
- **CDM subhalo** → localised flux anomalies at specific arc positions
- **Axion vortex** → periodic interference pattern around the ring

The PINN exploits this by:
1. Predicting the SIS ring parameters $(c_x, c_y, r_E)$ from image features
2. Generating a differentiable **ring template** $T$ from those parameters
3. Computing the **residual** $R = I_{01} - T$ (physics-unexplained content)
4. Using $R$ as additional discriminative features for classification

---

## 2. Architecture: PINNClassifier

```
Input I (1 × 150 × 150)
  └─ ResNet-18 backbone (pretrained, 1-ch adapted)
       → 512-dim feature vector  [global image representation]
  └─ Physics head: Linear(512→128→3)
       cx = tanh(·)·0.2          lens centroid x ∈ [−0.2, +0.2]
       cy = tanh(·)·0.2          lens centroid y ∈ [−0.2, +0.2]
       rE = sigmoid(·)·0.35+0.05 Einstein radius ∈ [0.05, 0.40]
  └─ Ring template T (differentiable SIS):
       T[x,y] = exp(−½·((||(x,y)−(cx,cy)||−rE)/σ)²),  σ=0.03
  └─ Residual encoder: CNN(I₀₁ − T) → 128-dim
       Conv(1→32, s=2) → Conv(32→64, s=2) → Conv(64→128, s=2) → GAP
  └─ Classifier: Linear(512+128→256→3)
```

**Loss function:**

$$\mathcal{L} = \mathcal{L}_{cls} + \lambda \cdot \mathcal{L}_{phys}$$

$$\mathcal{L}_{phys} = \|T - \text{GaussBlur}(I_{01})\|^2_2$$

$\mathcal{L}_{phys}$ penalises ring parameters that contradict the observed smooth ring structure (blurred image removes fine substructure noise).


In [ ]:
import os, sys
import torch
import numpy as np

sys.path.insert(0, os.path.join('Specfic_Test_VII', 'src'))

from model import PINNClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = PINNClassifier(
    in_channels=1, num_classes=3, img_size=150,
    ring_sigma=0.03, phys_lambda=0.1,
    dropout=0.3, pretrained=False)

print(f'Total parameters     : {model.count_all_parameters():,}')
print(f'Backbone (ResNet-18) : {sum(p.numel() for p in model.backbone.parameters()):,}')
print(f'Physics head         : {sum(p.numel() for p in model.physics_head.parameters()):,}')
print(f'Residual encoder     : {sum(p.numel() for p in model.residual_encoder.parameters()):,}')
print(f'Classifier head      : {sum(p.numel() for p in model.classifier.parameters()):,}')

# Demo forward pass
x = torch.randn(4, 1, 150, 150)
logits, ring, blur_tgt = model(x)
print(f'\nLogits:       {logits.shape}')
print(f'Ring template:{ring.shape}')
print(f'Blur target:  {blur_tgt.shape}')
print(f'Ring value range: [{ring.min():.3f}, {ring.max():.3f}]')


## 3. Dataset & Preprocessing

Same preprocessing as Common Test I and Test IV:
- **Log-stretch**: `log1p(x·10)/log1p(10)` → amplifies faint substructure
- **Normalise**: `(x − 0.5)/0.5` → zero-centred in `[−1, 1]`
- **Split**: merged train/val dirs, re-split 90:10 stratified (seed=42)


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('Specfic_Test_VII', 'src'))
from train import get_split_paths
from collections import Counter

DATASET_ROOT = os.path.normpath(os.path.join('Common_Test_I', 'dataset', 'dataset'))
train_items, val_items = get_split_paths(DATASET_ROOT, val_frac=0.10)
print(f'Train: {len(train_items):,}  |  Val: {len(val_items):,}')
print('Train class counts:', Counter(lbl for _, lbl in train_items))
print('Val   class counts:', Counter(lbl for _, lbl in val_items))


## 4. Training

| Phase | Epochs | Backbone | LR |
|-------|--------|----------|----|
| 1 — warm-up  | 20 | Frozen   | 1e-3 (head only) |
| 2 — fine-tune | 80 | Unfrozen | 1e-4 (backbone) / 5e-4 (rest) |

Physics loss weight: λ = 0.1
Early stopping: patience 10 (phase 1) / 20 (phase 2)


In [ ]:
import os, sys
sys.path.insert(0, os.path.join('Specfic_Test_VII', 'src'))

weights_path = os.path.join('Specfic_Test_VII', 'weights', 'best_pinn.pth')

if os.path.exists(weights_path):
    print(f'Weights found — skipping training.')
else:
    from train import main
    main()


## 5. Evaluation Results

In [ ]:
import os, sys, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sys.path.insert(0, os.path.join('Specfic_Test_VII', 'src'))
from model    import PINNClassifier
from train    import get_split_paths, LensingDataset
from evaluate import get_predictions, tta_predictions, plot_roc, plot_confusion, plot_confidence, plot_training_curves, plot_ring_examples
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import evaluate as ev7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATASET_ROOT = os.path.normpath(os.path.join('Common_Test_I', 'dataset', 'dataset'))
ev7.PLOTS_DIR = os.path.join('Specfic_Test_VII', 'plots')
os.makedirs(ev7.PLOTS_DIR, exist_ok=True)

_, val_items = get_split_paths(DATASET_ROOT, val_frac=0.10)
val_ds = LensingDataset(val_items, augment=False)
loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)

model = PINNClassifier(
    in_channels=1, num_classes=3, img_size=150,
    ring_sigma=0.03, dropout=0.3, pretrained=False)
model.load_state_dict(torch.load(
    os.path.join('Specfic_Test_VII', 'weights', 'best_pinn.pth'),
    map_location=device, weights_only=True))
model.to(device).eval()

CLASS_NAMES = ['No Substructure', 'CDM Subhalo', 'Axion Vortex']

labels, preds, probs = get_predictions(model, loader, device)
acc = (labels == preds).mean()
print(f'Standard Accuracy: {acc*100:.2f}%')
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))


In [ ]:
lbl_tta, prd_tta, prb_tta = tta_predictions(model, loader, device)
acc_tta = (lbl_tta == prd_tta).mean()
print(f'8x TTA Accuracy: {acc_tta*100:.2f}%')
print(classification_report(lbl_tta, prd_tta, target_names=CLASS_NAMES, digits=4))


In [ ]:
# Generate all plots
roc_std = plot_roc(labels,   probs,   'Standard')
roc_tta = plot_roc(lbl_tta,  prb_tta, 'TTA')
plot_confusion(labels,  preds,   'Standard')
plot_confusion(lbl_tta, prd_tta, 'TTA')
plot_confidence(labels, preds, probs)
plot_ring_examples(model, loader, device)
plot_training_curves()

print(f'Macro AUC (standard): {roc_std["macro"]:.4f}')
print(f'Macro AUC (TTA):      {roc_tta["macro"]:.4f}')


In [ ]:
# Display ring overlay examples (physics visualisation)
path = os.path.join('Specfic_Test_VII', 'plots', 'ring_examples.png')
if os.path.exists(path):
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img); ax.axis('off')
    ax.set_title('Predicted Einstein Ring Overlays (red contour = T=0.5 isoline)', fontsize=12)
    plt.tight_layout(); plt.show()


In [ ]:
# Display ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, fname, title in zip(
        axes,
        ['roc_curves_standard.png', 'roc_curves_tta.png'],
        ['Standard', '8× TTA']):
    path = os.path.join('Specfic_Test_VII', 'plots', fname)
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path)); ax.axis('off'); ax.set_title(title)
plt.tight_layout(); plt.show()


## 6. Results Summary

| Metric | Standard | 8× TTA |
|--------|----------|--------|
| Accuracy | ~96% | ~97% |
| Macro AUC | ~0.993 | ~0.995 |

### Comparison across tests

| Model | Accuracy (TTA) | Macro AUC (TTA) |
|-------|---------------|-----------------|
| ResNet-18 (Common Test I) | 96.51% | 0.9956 |
| HybridFNO (Test IV/Neural Operators) | ~97.2% | ~0.996 |
| **PINN (this test)** | **~97%** | **~0.995** |

---

## 7. Physics Interpretation

The PINN provides **interpretable predictions** beyond just class labels:

| Output | Meaning |
|--------|---------|
| `cx, cy` | Predicted lens centroid position |
| `rE` | Predicted Einstein radius |
| Ring template `T` | Expected smooth lensing signal (SIS model) |
| Residual `I − T` | Physics-unexplained signal → substructure signature |

**For "No Substructure"**: residuals ≈ 0 → smooth ring matches perfectly
**For "CDM Subhalo"**: residuals show localised bright/dark spots at subhalo positions
**For "Axion Vortex"**: residuals show periodic fringe patterns

The physics loss $\mathcal{L}_{phys}$ enforces that the predicted ring is consistent with the actual image, making the model's physics parameters physically meaningful rather than arbitrary latent dimensions.

### How it differs from standard CNNs

| Property | Standard CNN | PINN (this work) |
|----------|-------------|-----------------|
| Architecture | Black-box feature extractor | Physics prior + residual |
| Interpretability | None | Einstein ring parameters |
| Physics constraint | None | Thin lens equation in loss |
| Residual features | N/A | Explicit substructure signal |
